<a href="https://colab.research.google.com/github/akankshasinhagithub/Niramaya-RAG-Demo/blob/main/NIRAMAYA_RAG_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval-Augmented Generation (RAG) with LangChain and GPT-4: A Simple Tutorial

# NIRAMAYA: A Simple RAG Prototype using GPT-4 and LangChain

This notebook demonstrates a lightweight Retrieval-Augmented Generation (RAG) pipeline using LangChain and OpenAI's GPT-4 to answer health-related questions based on a small curated knowledge base.

Inspired by the lessons of the COVID-19 pandemic, NIRAMAYA is a vision for empathetic, trustworthy AI that supports collective wellbeing during crises.

We show:
- How to build a basic RAG system
- Why it matters for healthcare
- How this fits into a broader AI-for-good initiative


## 1. Install Required Libraries
We install LangChain, FAISS, and OpenAI dependencies.


In [1]:
!pip install -q langchain openai faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 39.2 MB/s eta 0:00:00


In [6]:
!pip install -U langchain langchain-community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.0 MB/s eta 0:00:00


In [9]:
!pip install tiktoken


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.2 MB/s eta 0:00:00


# Setup: Environment and API Keys

In [7]:
from langchain.embeddings.openai import OpenAIEmbeddings


In [2]:
import os, getpass


In [3]:
# Set OpenAI API key – you will be prompted to enter it securely
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key:")

Enter your OpenAI API key:··········


## 2. Define Knowledge Base
We use 3 short texts simulating pandemic FAQs (symptoms, vaccines, mental health).


In [4]:
# Define our small knowledge base as a list of documents (texts)
docs = [
    """COVID-19 Symptoms:
COVID-19 is an infectious disease caused by the coronavirus SARS-CoV-2.
Common symptoms include **fever**, **dry cough**, **tiredness**, and **loss of taste or smell**.
Some patients also experience aches, nasal congestion, runny nose, sore throat, or diarrhea.
Severe symptoms (less common) can include difficulty breathing or shortness of breath, chest pain, and loss of speech or movement.
Symptoms typically appear 2–14 days after exposure to the virus.""",

    """COVID-19 Vaccines FAQ:
COVID-19 vaccines help protect people from getting sick with COVID-19.
They work by training the body's immune system to recognize and fight the coronavirus.
The vaccines underwent rigorous testing in clinical trials and have been proven to be **safe and effective**.
**Common side effects** of the COVID-19 vaccines are mild and short-lived.
These side effects can include a sore arm at the injection site, fatigue, headache, muscle or joint pain, chills, and fever.
Serious side effects are very rare. Health authorities recommend vaccination for eligible people to reduce the risk of severe illness.""",

    """Mental Health Tips:
The COVID-19 pandemic has been challenging, and it's normal to feel stress or anxiety during this time.
To manage stress, try to maintain a regular routine, get enough sleep and exercise, and eat healthy meals.
Stay **connected** with friends and family via phone or video calls, even if you cannot meet in person.
Practicing relaxation techniques like deep breathing, meditation, or yoga can help calm your mind.
Limit your exposure to pandemic news if it makes you anxious.
If you feel overwhelmed or persistently sad, consider seeking help from a mental health professional. Remember, it's okay to ask for support."""
]
print(f"Loaded {len(docs)} documents in the knowledge base.")


Loaded 3 documents in the knowledge base.


## 3. Embed and Index
Each document is embedded using OpenAI's `text-embedding-ada-002` model and stored in a FAISS vector index.


In [10]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS

# Initialize the OpenAI embeddings module (this will use the API key set earlier)
embedding_model = OpenAIEmbeddings()  # uses text-embedding-ada-002 by default

# Create the FAISS vector store from our list of documents
vector_store = FAISS.from_texts(docs, embedding_model)

print("Documents embedded and indexed in vector store.")


Documents embedded and indexed in vector store.


## 4. Setting Up the Retrieval QA Chain

In [11]:
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA

# Initialize the LLM (GPT-4) for generating answers
llm = ChatOpenAI(model_name="gpt-4", temperature=0)

# Create a retriever from the vector store (we can specify search params like top_k if desired)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})  # we'll retrieve top-2 docs for each query

# Set up the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",      # "stuff" simply dumps all retrieved docs into the prompt (the simplest method)
    retriever=retriever,
    return_source_documents=False   # we don't need the source docs returned for this demo
)


<ipython-input-11-55339f2c2b86>:5: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model_name="gpt-4", temperature=0)


## 5. Run Retrieval-Augmented QA
We ask GPT-4 questions grounded in the most relevant retrieved documents.


Let's test our retrieval-augmented QA system with a few example queries:

1) “What are the symptoms of COVID-19?” – This should retrieve the document about COVID-19 symptoms and have GPT-4 summarize the key symptoms.

2) “What are common side effects of COVID-19 vaccines?” – This should fetch information from the vaccine FAQ document and answer with those side effects.

3) “How can I manage stress during the pandemic?” – This should bring in the mental health tips and have the model list some coping strategies.
We'll run these queries through the chain and print the answers:

In [12]:
# Define some example questions to ask our RAG system
queries = [
    "What are the symptoms of COVID-19?",
    "What are common side effects of COVID-19 vaccines?",
    "How can I manage stress during the COVID-19 pandemic?"
]

for i, query in enumerate(queries, start=1):
    print(f"Question {i}: {query}")
    answer = qa_chain.run(query)
    print(f"Answer: {answer}\n")


Question 1: What are the symptoms of COVID-19?


<ipython-input-12-3b8120e30cff>:10: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  answer = qa_chain.run(query)


Answer: The common symptoms of COVID-19 include fever, dry cough, tiredness, and loss of taste or smell. Some patients may also experience aches, nasal congestion, runny nose, sore throat, or diarrhea. Less common but severe symptoms can include difficulty breathing or shortness of breath, chest pain, and loss of speech or movement. Symptoms typically appear 2–14 days after exposure to the virus.

Question 2: What are common side effects of COVID-19 vaccines?
Answer: Common side effects of the COVID-19 vaccines can include a sore arm at the injection site, fatigue, headache, muscle or joint pain, chills, and fever. These side effects are usually mild and short-lived.

Question 3: How can I manage stress during the COVID-19 pandemic?
Answer: To manage stress during the COVID-19 pandemic, you can maintain a regular routine, get enough sleep and exercise, and eat healthy meals. It's also beneficial to stay connected with friends and family via phone or video calls. Practicing relaxation t

## 🔚 Conclusion: Why This Matters

This notebook is more than a technical demo — it’s a proof of concept for NIRAMAYA, a vision of using Retrieval-Augmented Generation (RAG) to build AI companions that offer trustworthy, context-aware support during health crises.

While this demo is simple, it lays the groundwork for:
- Grounded AI in healthcare, mental wellbeing, and public awareness
- Future integration with multilingual documents and richer retrieval
- Scalable systems that combine empathy with accuracy

Whether it's a pandemic, climate disaster, or local health emergency, **AI should serve humans — not overwhelm them.** That’s the mission behind NIRAMAYA.
